# LLM APIs Learning Notebook
Self-study notes and code for learning how to work with LLM APIs (using Groq, OpenAI-compatible).

## Step 1: Setup & Authentication

**Main points:**
- API keys should never be hardcoded directly in your script
- Store the key in a `.env` file and load it with `python-dotenv`
- Install dependencies: `pip install groq python-dotenv`

In [ ]:
# Setup & Authentication
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Quick check that the key loaded correctly
key = os.getenv("GROQ_API_KEY")
print("Key found:", key is not None)

## Step 2: First API Call

**Main points:**
- The API is request/response based — you send messages, you get a response object back
- `response.choices[0].message.content` is where the actual reply text lives
- The full response object also contains token usage, model used, and finish reason

In [ ]:
# First API Call
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in 5 words."}]
)

print(response.choices[0].message.content)

## Step 3: Understanding Roles

**Main points:**
- `system` sets the model's behavior/persona for the whole conversation
- `user` is the human's input
- `assistant` is the model's own previous replies (used when building conversation history)
- Changing the system message is the main lever for controlling tone and behavior

In [ ]:
# Understanding Roles
messages = [
    {"role": "system", "content": "You only respond in pirate speak."},
    {"role": "user", "content": "What's the weather like?"}
]

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages
)

print(response.choices[0].message.content)

## Step 4: Multi-turn Conversations (Memory)

**Main points:**
- The API itself is stateless — it has no memory between calls
- "Memory" is achieved by resending the entire conversation history with every call
- Each new user message and each assistant reply gets appended to the same list

In [ ]:
# Multi-turn Conversations (Memory)
messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

def chat(user_input):
    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
    )

    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat("My favorite programming language is Python."))

In [ ]:
# Test that memory actually works
print(chat("What did I just tell you?"))

## Step 5: Controlling Output (Temperature & Max Tokens)

**Main points:**
- `temperature` controls randomness: 0 = deterministic/focused, 1.5+ = more random/creative
- `max_tokens` caps the response length — if too low, output gets cut off mid-sentence
- `finish_reason` tells you why the model stopped: `"stop"` = natural finish, `"length"` = hit the max_tokens cap

In [ ]:
# Temperature comparison
prompt = "Write one sentence about the ocean."

response_low = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=0
)

response_high = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    temperature=1.5
)

print("TEMP 0:", response_low.choices[0].message.content)
print("\nTEMP 1.5:", response_high.choices[0].message.content)

In [ ]:
# Max tokens comparison
prompt = "Explain how photosynthesis works."

response_full = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=300
)

response_cut = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=15
)

print("FULL:", response_full.choices[0].message.content)
print("\nCUT OFF:", response_cut.choices[0].message.content)
print("\nFinish reason (cut):", response_cut.choices[0].finish_reason)

## Step 6: Structured Outputs (JSON)

**Main points:**
- `response_format={"type": "json_object"}` ensures the output is valid JSON
- You still need to explicitly instruct the model in the prompt to respond only in JSON
- Always parse the result with `json.loads()` rather than trusting raw text

In [ ]:
# Structured Outputs (JSON)
import json

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": 'Respond only with valid JSON: {"name": str, "age": int}'},
        {"role": "user", "content": "John is 25 years old"}
    ],
    response_format={"type": "json_object"}
)

data = json.loads(response.choices[0].message.content)
print(data["name"], data["age"])

## Step 7: Error Handling

**Main points:**
- API calls can fail: invalid keys, rate limits, network issues, malformed JSON responses
- Wrap calls in `try/except` so failures don't crash your whole program
- When expecting JSON, wrap the `json.loads()` call separately so you can catch parsing failures too

In [ ]:
# Error Handling
try:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "Hello"}]
    )
    print(response.choices[0].message.content)
except Exception as e:
    print(f"API error: {e}")

In [ ]:
# Handling JSON parsing failures specifically
raw_text = response.choices[0].message.content

try:
    data = json.loads(raw_text)
except json.JSONDecodeError:
    print("Model did not return valid JSON. Raw output:")
    print(raw_text)
    data = None

## Step 8: Porting to OpenAI

**Main points:**
- OpenAI's API has nearly identical structure to Groq's — same `chat.completions.create()` pattern
- Swapping providers mainly means changing the import, the client, and the model name
- Confirms whether you learned the *pattern* rather than just one SDK

In [ ]:
# Porting to OpenAI (structure only — requires an OpenAI API key)
from openai import OpenAI

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello in 5 words."}]
)

print(response.choices[0].message.content)